<a href="https://colab.research.google.com/github/BuyWhere/llama_index/blob/main/docs/examples/cookbooks/buywhere_product_search_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Product Search Agent with BuyWhere + LlamaIndexBuild an AI shopping agent that searches real product catalogs across Shopee, Lazada, Amazon, Walmart, and 20+ platforms — all from a single LlamaIndex agent.This cookbook shows you how to:- Search 1.5M+ products by keyword, category, price, and region- Find the best deals with discount filtering- Browse available categories- Build a ReAct agent that chains these tools together

In [ ]:
%pip install llama-index%pip install llama-index-llms-openai%pip install httpx

## Prerequisites1. Get a free BuyWhere API key at [https://api.buywhere.ai/v1/auth/register](https://api.buywhere.ai/v1/auth/register)2. Set it as an environment variable:```pythonimport osos.environ["BUYWHERE_API_KEY"] = "bw_live_xxxxx"```> **Note:** For production, replace with your real key. The API is free to get started.

In [ ]:
import osimport jsonimport httpxBUYWHERE_API_KEY = os.environ.get("BUYWHERE_API_KEY", "")MCP_SERVER_URL = os.environ.get("BUYWHERE_MCP_URL", "https://api.buywhere.ai/mcp")if not BUYWHERE_API_KEY:    print("WARNING: BUYWHERE_API_KEY not set. Set it above with your real key.")else:    print(f"BuyWhere API key found ({BUYWHERE_API_KEY[:8]}...{BUYWHERE_API_KEY[-4:]})")

## Define BuyWhere Tool FunctionsThese functions wrap the BuyWhere REST API. In production, you'd use the `llama-index-tools-buywhere` package, but the raw API calls work great for this example.

In [ ]:
def _api_get(path, params=None):    headers = {"Accept": "application/json"}    if BUYWHERE_API_KEY:        headers["Authorization"] = f"Bearer {BUYWHERE_API_KEY}"    url = f"https://api.buywhere.ai{path}"    resp = httpx.get(url, params=params, headers=headers, timeout=30.0)    resp.raise_for_status()    return resp.json()def search_products(query, category=None, min_price=None, max_price=None,                    source=None, market=None, limit=10):    """Search the BuyWhere product catalog by keyword."""    params = {"q": query, "limit": min(limit, 50)}    for key, val in (("category", category), ("min_price", min_price),                     ("max_price", max_price), ("source", source),                     ("country", market)):        if val is not None:            params[key] = val    try:        data = _api_get("/v1/products/search", params)    except httpx.HTTPStatusError as exc:        if exc.response.status_code == 401:            return "Authentication failed: Check your BUYWHERE_API_KEY"        if exc.response.status_code == 403:            return "Access denied: Your API key may not have permission"        return f"Search failed: {exc}"    except httpx.RequestError as exc:        return f"Network error: Could not reach BuyWhere API — {exc}"    items = data.get("items", []) if isinstance(data, dict) else []    if not items:        return f"No products found for: {query}"    lines = [f"Found {len(items)} product(s) for **{query}**:\n"]    for i, p in enumerate(items, 1):        name = p.get("name") or p.get("title") or "Unknown"        price = _fmt_price(p.get("price"), p.get("currency", "SGD"))        src = p.get("source", "unknown")        pid = p.get("id", "")        url = p.get("affiliate_url") or p.get("buy_url") or ""        url_line = f"\n   URL: {url}" if url else ""        lines.append(f"{i}. **{name}**\n   Price: {price} | Platform: {src}{url_line}\n   ID: {pid}")    return "\n".join(lines)def get_deals(category=None, min_discount_pct=10, limit=10):    """Find products with significant price drops."""    params = {"min_discount_pct": min_discount_pct, "limit": min(limit, 50)}    if category:        params["category"] = category    try:        data = _api_get("/v1/products/deals", params)    except httpx.HTTPStatusError as exc:        if exc.response.status_code == 401:            return "Authentication failed: Check your BUYWHERE_API_KEY"        return f"Deals fetch failed: {exc}"    except httpx.RequestError as exc:        return f"Network error: Could not reach BuyWhere API — {exc}"    items = data.get("items", []) if isinstance(data, dict) else []    if not items:        return f"No deals found with >={min_discount_pct}% discount."    lines = [f"Found {len(items)} deal(s) with >={min_discount_pct}% discount:\n"]    for i, d in enumerate(items, 1):        current = _fmt_price(d.get("price"), d.get("currency", "SGD"))        original = _fmt_price(d.get("original_price"), d.get("currency", "SGD")) if d.get("original_price") else "N/A"        discount = d.get("discount_pct", 0) or 0        lines.append(f"{i}. **{d.get('name', 'Unknown')}**\n"                     f"   Current: {current} | Was: {original} | -{discount}%\n"                     f"   Platform: {d.get('source', 'unknown')} | ID: {d.get('id', '')}")    return "\n".join(lines)def list_categories():    """List all product categories with product counts."""    try:        data = _api_get("/v1/categories")    except httpx.HTTPStatusError as exc:        if exc.response.status_code == 401:            return "Authentication failed: Check your BUYWHERE_API_KEY"        return f"Categories fetch failed: {exc}"    except httpx.RequestError as exc:        return f"Network error: Could not reach BuyWhere API — {exc}"    categories = data.get("categories", []) if isinstance(data, dict) else []    total = data.get("total", len(categories))    if not categories:        return "No categories found."    lines = [f"**{total} categories available:**\n"]    for cat in categories:        lines.append(f"- {cat.get('name', 'Unknown')} ({cat.get('count', 0):,} products)")    return "\n".join(lines)def _fmt_price(price, currency="SGD"):    if price is None:        return "N/A"    try:        return f"{currency} {float(price):.2f}"    except (TypeError, ValueError):        return str(price)print("All tool functions defined successfully.")

## Try the ToolsLet's run each tool independently to see what the BuyWhere catalog looks like.

In [ ]:
# 1. Browse categoriesprint(">>> Available Categories")print(list_categories())

In [ ]:
# 2. Search for productsprint(">>> Searching for wireless headphones in SG")print(search_products("wireless headphones", market="SG", limit=3))

In [ ]:
# 3. Find dealsprint(">>> Deals with 15%+ discount in Electronics")print(get_deals(category="Electronics", min_discount_pct=15, limit=5))

## Build a ReAct Agent with BuyWhere ToolsNow let's wrap these tools into a LlamaIndex ReAct agent. The agent can reason about which tool to call based on the user's request.

In [ ]:
from llama_index.core.agent import ReActAgentfrom llama_index.llms.openai import OpenAIfrom llama_index.core.tools import FunctionTool# Wrap each function as a LlamaIndex toolsearch_tool = FunctionTool.from_defaults(    fn=search_products,    name="search_products",    description="Search products by keyword with optional category, price range, source, and region filters",)deals_tool = FunctionTool.from_defaults(    fn=get_deals,    name="get_deals",    description="Find discounted products with optional category and minimum discount filter",)categories_tool = FunctionTool.from_defaults(    fn=list_categories,    name="list_categories",    description="List all available product categories with product counts",)llm = OpenAI(model="gpt-4o")agent = ReActAgent.from_tools(    [search_tool, deals_tool, categories_tool],    llm=llm,    verbose=True,)print("Agent ready!")

## Chat with the AgentAsk the agent to help you shop. It will reason about which tools to call.

In [ ]:
response = agent.chat(    "What categories are available? Then find me wireless headphones "    "under SGD 50 in Singapore.")print(response)

In [ ]:
response = agent.chat(    "Any good deals on electronics right now? I'm looking for "    "discounts of at least 20%.")print(response)

## Error HandlingThe tools handle common errors gracefully:| Scenario | Response ||----------|----------|| Missing/invalid API key | Clear authentication error message || Network timeout | Network error with details || No results | Informative empty result message || Rate limiting | Handled with retry guidance |

## Next Steps- **Use the official package:** Once available, `pip install llama-index-tools-buywhere` for a more streamlined integration- **Connect via MCP:** BuyWhere also supports the Model Context Protocol — connect any MCP-compatible agent directly- **Explore the API:** Full API reference at [https://api.buywhere.ai/docs](https://api.buywhere.ai/docs)- **Run yourself:** Get your free API key at [https://api.buywhere.ai/v1/auth/register](https://api.buywhere.ai/v1/auth/register)